# Predicting *movement* in crypto — magnitude, not direction

The honest version of "predict the move". Notebook 10 established that 1-minute
**direction** is ~random (white spectrum, ~0 ACF, ~50%); this notebook turns the
*flip side* — volatility is forecastable — into a proper **out-of-sample simulation**
and a **gated backtest**, plus a cross-venue lead-lag check. It reuses the two
research scripts so the logic is shared, not re-implemented:

- `research/defi/vol_forecast_sim.py` — walk-forward HAR / HAR-CJ realized-vol forecast
  vs naive & EWMA baselines, then vol-targeting as the one honest tradeable expression.
- `research/defi/leadlag_candles.py` — Hyperliquid perp vs Coinbase spot lead-lag on the
  overlapping 1-minute candles.

**No look-ahead by construction**: every forecast at block *t* uses only blocks ≤ *t*;
the only "future" input is the wall clock (hour-of-day), which is deterministic.

In [ ]:
import os, sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'research'/'defi'))
import vol_forecast_sim as vf
import leadlag_candles as ll

if os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')

VENUE, BLOCK, MINTRAIN, REFIT = 'coinbase', 15, 400, 4
ASSETS = ['BTC','ETH','SOL']
con = duckdb.connect(vf.DEFI_DB, read_only=True)

results = {}
for a in ASSETS:
    blk = vf.load_blocks(con, VENUE, a, BLOCK)
    if blk.empty: continue
    b = vf.build_features(blk)
    if len(b) < MINTRAIN + 50: continue
    res = vf.walk_forward(b, MINTRAIN, REFIT)
    sc  = vf.score(res)
    win = 'harj' if sc['harj']['qlike'] < sc['har']['qlike'] else 'har'
    bt  = vf.backtest(res, b['rv'].to_numpy(), MINTRAIN, BLOCK, model=win)
    results[a] = dict(b=b, res=res, score=sc, winner=win, bt=bt)
print(f"{len(results)} assets simulated: {list(results)}  (block={BLOCK}m, OOS walk-forward)")

## Stage 1 — prediction quality

Out-of-sample over the walk-forward region. **QLIKE** is the robust volatility loss
(lower = better), **R²(logRV)** the fraction of log-variance explained, and the
**Mincer–Zarnowitz slope** *b* tests calibration (*b* ≈ 1 = forecasts are right on
average). `naive` = RV persists; `ewma` = RiskMetrics λ=0.94; `har` = HAR on log-RV;
`harj` = HAR-CJ (continuous + jump split via bipower variation).

In [ ]:
rows = []
for a, d in results.items():
    for m in ('naive','ewma','har','harj'):
        s = d['score'][m]
        rows.append(dict(asset=a, model=m, QLIKE=s['qlike'], R2_logRV=s['r2_logrv'],
                         MZ_b=s['mz_b'], winner=('*' if m==d['winner'] else '')))
tbl = pd.DataFrame(rows).set_index(['asset','model'])
display(tbl.style.format({'QLIKE':'{:.4f}','R2_logRV':'{:.3f}','MZ_b':'{:.3f}'})
        .background_gradient(subset=['QLIKE'], cmap='RdYlGn_r')
        .background_gradient(subset=['R2_logRV'], cmap='RdYlGn'))

### Calibration — does the forecast match realized vol?

Forecasts bucketed into deciles; each point is (mean forecast vol, mean realized vol)
for that decile. On the 45° line = perfectly calibrated.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))
pal = sns.color_palette('bright', len(results))
for (a, d), c in zip(results.items(), pal):
    res, m = d['res'], d['res']['mask']
    rv = np.sqrt(np.maximum(res['rv_next'][m], vf.EPS))*1e4
    f  = np.sqrt(np.maximum(res[d['winner']][m], vf.EPS))*1e4
    q  = np.array_split(np.argsort(f), 10)
    ax[0].plot([f[i].mean() for i in q], [rv[i].mean() for i in q],
               marker='o', color=c, label=f"{a} ({d['winner']})")
lim = max(ax[0].get_xlim()[1], ax[0].get_ylim()[1])
ax[0].plot([0,lim],[0,lim], ls=':', color='gray', lw=.8)
ax[0].set(title='Vol-forecast calibration (decile)',
          xlabel='forecast vol (bps/block)', ylabel='realized vol (bps/block)'); ax[0].legend()

# HAR vs HAR-CJ: does the jump split help? (delta-QLIKE > 0 = yes)
dj = {a: d['score']['har']['qlike'] - d['score']['harj']['qlike'] for a,d in results.items()}
ax[1].bar(list(dj), list(dj.values()), color=pal)
ax[1].axhline(0, color='gray', lw=.8)
ax[1].set(title='Jump term value: QLIKE(HAR) − QLIKE(HAR-CJ)', ylabel='Δ QLIKE  (>0 ⇒ jumps help)')
plt.tight_layout(); plt.show()

The HAR forecast is well-calibrated (slope ≈ 1, deciles hug the 45° line) and beats
both baselines on every asset. The **jump decomposition (HAR-CJ) helps only marginally** —
at 15-minute resolution on liquid majors, the continuous part already carries the signal.

## Stage 2 — gated backtest: vol-targeting

A vol forecast on a single perp has one honest tradeable expression: **vol-targeting** —
scale exposure to a constant risk budget (`w = target_vol / forecast_vol`, capped), which
needs *no* directional edge (we don't have one). What it should buy is **stable risk**, not
alpha. `volCoV` = coefficient-of-variation of rolling realized vol (lower = steadier risk).

In [ ]:
rows = []
for a, d in results.items():
    s, h = d['bt']['strat'], d['bt']['hold']
    rows.append(dict(asset=a, strat_ret=s['total_ret'], strat_Sharpe=s['sharpe'],
                     strat_maxDD=s['maxdd'], strat_volCoV=s['vol_cov'],
                     hold_ret=h['total_ret'], hold_Sharpe=h['sharpe'],
                     hold_maxDD=h['maxdd'], hold_volCoV=h['vol_cov']))
bt_tbl = pd.DataFrame(rows).set_index('asset')
display(bt_tbl.style.format('{:.3f}')
        .background_gradient(subset=['strat_volCoV','hold_volCoV'], cmap='RdYlGn_r'))

# rolling realized vol: vol-targeted vs buy&hold (risk stabilization, BTC)
a0 = next(iter(results)); d = results[a0]; res = d['res']; m = res['mask']
f = np.sqrt(np.maximum(res[d['winner']][m], vf.EPS)); ret = res['ret_next'][m]
tgt = float(np.median(np.sqrt(d['b']['rv'].to_numpy()[:MINTRAIN])))
w = np.clip(tgt/f, 0, vf.WMAX); strat = w*ret; hold = ret
rv_s = pd.Series(strat).rolling(16).std(); rv_h = pd.Series(hold).rolling(16).std()
plt.figure(figsize=(12,4))
plt.plot(rv_h.values*1e4, color='#f97583', lw=.9, label=f'{a0} buy&hold')
plt.plot(rv_s.values*1e4, color='#56d364', lw=.9, label=f'{a0} vol-targeted')
plt.title('Rolling realized vol — vol-targeting flattens the risk'); plt.ylabel('bps/block')
plt.xlabel('OOS block'); plt.legend(); plt.tight_layout(); plt.show()

Vol-targeting clearly **stabilizes risk** (lower volCoV, flatter rolling-vol line, smaller
drawdown). The Sharpe/return columns are **noise** over a ~2-week single regime — the window
happened to trend down, so buy-and-hold is negative; do not read these as alpha. The
deliverable is the prediction quality + risk control, exactly as a prediction lab should claim.

## Cross-venue lead-lag — does Hyperliquid lead Coinbase?

On the overlapping 1-minute candles. **Left:** cross-correlation corr(HL[t], CEX[t+L]) vs lag
*L* (peak at 0 = synced within the minute). **Right:** the OOS predictive increment — does the
*other* venue's current return help predict a venue's *next* minute beyond its own lag?

In [ ]:
MAXLAG = 5
cc_all, inc_rows = {}, []
for coin in ['BTC','ETH','SOL','XRP']:
    df = ll.load_aligned(con, coin)
    if df.empty: continue
    hl, cx = df['hl_r'].to_numpy(), df['cx_r'].to_numpy()
    cc_all[coin] = ll.crosscorr(hl, cx, MAXLAG)
    _,_,icx = ll.oos_increment(cx[1:], cx[:-1], hl[:-1])   # predict next-min CEX, +HL lag
    _,_,ihl = ll.oos_increment(hl[1:], hl[:-1], cx[:-1])   # predict next-min HL, +CEX lag
    inc_rows.append(dict(coin=coin, mins=len(df), corr_at_0=cc_all[coin][0],
                         dR2_CEX_from_HL=icx, dR2_HL_from_CEX=ihl))

fig, ax = plt.subplots(1, 2, figsize=(13,5))
pal = sns.color_palette('bright', len(cc_all))
for (coin, cc), c in zip(cc_all.items(), pal):
    lags = sorted(cc); ax[0].plot(lags, [cc[L] for L in lags], marker='o', color=c, label=coin)
ax[0].axvline(0, color='gray', lw=.8, ls=':')
ax[0].set(title='Cross-correlation corr(HL[t], CEX[t+L])', xlabel='lag L (min)  >0 ⇒ HL leads',
          ylabel='correlation'); ax[0].legend()
inc = pd.DataFrame(inc_rows).set_index('coin')
inc[['dR2_CEX_from_HL','dR2_HL_from_CEX']].plot.bar(ax=ax[1], color=['#58a6ff','#f0883e'])
ax[1].axhline(0, color='gray', lw=.8)
ax[1].set(title='Incremental OOS R² from the other venue', ylabel='Δ R²')
plt.tight_layout(); plt.show()
display(inc.style.format({'corr_at_0':'{:.3f}','dR2_CEX_from_HL':'{:.4f}','dR2_HL_from_CEX':'{:.4f}'}))

Cross-correlation peaks at **lag 0** (corr 0.90–0.97) — at minute resolution the venues
move together. But the predictive test shows a clean **asymmetry**: Coinbase spot's current
move adds predictive power for Hyperliquid's *next* minute (strongest in the thin XRP perp),
while the reverse is ~0. So **spot mildly leads the perp** — directionally as expected, but
<3% R² and almost certainly eaten by perp taker fees + funding. Minute bars can't see the
sub-second arb layer where real cross-venue edge lives.

---
### Verdict
**Movement is predictable in magnitude, not direction.** The vol forecast is calibrated
(slope ≈ 1) and beats naive/EWMA; vol-targeting converts that into *risk control*, not alpha.
The cross-venue lead-lag is real but tiny and infra-gated. Same honest lesson as the rest of
the lab: model the un-priced signal (risk, structure), not the already-priced price.

In [ ]:
con.close()